# Experiment 5: Geo-Visualization with Maps

**Aim:** Visualize sales by Indian region/state/city using the geography dimension table. City coordinates are included below for the cities in this dataset.

## Import libraries and load the supplied CSV files
The five dimension/fact tables are joined using their ID/date keys. Sales, cost and profit are calculated from the supplied unit price, unit cost, units sold and discount.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

DATA_DIR = Path("datasets")

def load_data():
    required_files = [
        "Dim_Customer.csv", "Dim_Date.csv", "Dim_Geography.csv",
        "Dim_Product.csv", "Fact_Orders.csv"
    ]
    missing = [f for f in required_files if not (DATA_DIR / f).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing CSV file(s): " + ", ".join(missing) +
            ". Put all five CSV files in the datasets folder beside this notebook."
        )
    customer = pd.read_csv(DATA_DIR / "Dim_Customer.csv")
    dates = pd.read_csv(DATA_DIR / "Dim_Date.csv")
    geography = pd.read_csv(DATA_DIR / "Dim_Geography.csv")
    product = pd.read_csv(DATA_DIR / "Dim_Product.csv")
    orders = pd.read_csv(DATA_DIR / "Fact_Orders.csv")

    # Join the fact table with dimension tables using their matching keys.
    data = orders.merge(customer, on="Customer_ID", how="left")
    data = data.merge(product, on="Product_ID", how="left")
    data = data.merge(geography, on="Geo_ID", how="left")
    data = data.merge(dates, left_on="Order_Date", right_on="Date", how="left")

    data["Order_Date"] = pd.to_datetime(data["Order_Date"], errors="coerce")
    data["Ship_Date"] = pd.to_datetime(data["Ship_Date"], errors="coerce")
    data["Sales"] = data["Units_Sold"] * data["Unit_Price"] * (1 - data["Discount_Pct"])
    data["Cost"] = data["Units_Sold"] * data["Unit_Cost"]
    data["Profit"] = data["Sales"] - data["Cost"]
    data["Profit_Margin"] = np.where(data["Sales"] != 0,
                                     data["Profit"] / data["Sales"] * 100, 0)
    data["Order_Month"] = data["Order_Date"].dt.to_period("M").astype(str)
    return data

## Python code and visualization

In [ ]:
df = load_data()

# Region-wise sales bar chart (all regions are in India).
region_sales = df.groupby("Region", as_index=False)["Sales"].sum()
px.bar(region_sales, x="Region", y="Sales", title="Region-wise Sales in India",
       text_auto=".2s").show()

# City-level symbol map using approximate city coordinates.
city_coordinates = pd.DataFrame({
    "City": ["New Delhi", "Noida", "Mumbai", "Pune", "Bengaluru",
             "Hyderabad", "Kolkata", "Indore"],
    "Latitude": [28.6139, 28.5355, 19.0760, 18.5204, 12.9716,
                 17.3850, 22.5726, 22.7196],
    "Longitude": [77.2090, 77.3910, 72.8777, 73.8567, 77.5946,
                  78.4867, 88.3639, 75.8577]
})
city_sales = df.groupby(["City", "State"], as_index=False)["Sales"].sum()
city_sales = city_sales.merge(city_coordinates, on="City", how="left")

fig = px.scatter_map(
    city_sales, lat="Latitude", lon="Longitude", size="Sales",
    color="Region", hover_name="City", hover_data=["State", "Sales"],
    zoom=3, center={"lat": 22.5, "lon": 79.0},
    title="City-wise Sales Map – India", map_style="open-street-map"
)
fig.show()

## Result
The experiment was performed using the supplied retail dataset. Run all cells to generate the tables, calculations and visualizations.